# Question 2: Regression Models Comparison

Dataset: **UCI Superconductivity** (21 263 samples, 81 features, target = `critical_temp` in Kelvin)

Models compared:
1. Linear Regression with Gradient Descent Optimization
2. Polynomial Regression (Degree 2 + Ridge, CV-tuned alpha)
3. Ridge Regression (L2, CV-tuned alpha)
4. Lasso Regression (L1, CV-tuned alpha)

## Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import SGDRegressor, Ridge, Lasso, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error

df = pd.read_csv(r'd:\study\college\5th semester\ML\CourseProject\Question2\superconductivty+data\train.csv')

In [ ]:
X = df.drop('critical_temp', axis=1)
y = df['critical_temp']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

Scaler = StandardScaler()   # scaling must be done before fitting the model to avoid bias in the model. Scaling is done to bring all the features to a similar scale, which helps in improving the performance of the model.
X_train_Scaled = Scaler.fit_transform(X_train)
X_test_Scaled = Scaler.transform(X_test)

Data Loaded: 21263 samples, 81 features.


## 1. Linear Regression with Gradient Descent Optimization

In [ ]:
# Linear regression with Gradient Descent optimization (SGDRegressor with squared loss and no penalty)
reg = SGDRegressor(loss='squared_error', penalty=None, max_iter=2000, tol=1e-4, random_state=42)
reg.fit(X_train_Scaled, y_train)

y_pred = reg.predict(X_test_Scaled)

In [ ]:
r2_linear = r2_score(y_test, y_pred)
mse_linear = mean_squared_error(y_test, y_pred)
rmse_linear = root_mean_squared_error(y_test, y_pred)

r2, mse, rmse = r2_linear, mse_linear, rmse_linear

print("Iterations to converge:", reg.n_iter_)
print("R2 Score:", r2_linear)
print("Test MSE:", mse_linear)
print("Root Mean Squared Error:", rmse_linear)

## 2. Polynomial Regression

In [ ]:
# 1. Build the pipeline
pipe = Pipeline([
    ("scale1", StandardScaler()),
    ("poly",   PolynomialFeatures(degree=2, include_bias=False)),
    ("scale2", StandardScaler()),
    ("ridge",  Ridge())
])

# 2. Alpha values: 0.01, 0.1, 1, 10, 100, 1000
alphas = [0.01, 0.1, 1, 10, 100, 1000]
param_grid = {"ridge__alpha": alphas}

In [ ]:
# 3. Cross-validation to pick the best alpha
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="r2", n_jobs=-1)
grid.fit(X_train, y_train)

In [ ]:
# 4. Results for each alpha
for a, score in zip(alphas, grid.cv_results_["mean_test_score"]):
    print(f"alpha={a:<8} CV R² = {score:.4f}")
print("Best alpha:", grid.best_params_["ridge__alpha"])

alpha=0.01     CV R² = 0.6779
alpha=0.1      CV R² = 0.8004
alpha=1        CV R² = 0.8426
alpha=10       CV R² = 0.8468
alpha=100      CV R² = 0.8374
alpha=1000     CV R² = 0.8154
Best alpha: 10


Look above we have done the cross-validation to select the regularization parameter(alpha), which 
comes out to be 10, 

In [ ]:
# 5. Final evaluation on unseen test data
pred_poly = grid.predict(X_test)
pred = pred_poly

r2_poly = r2_score(y_test, pred_poly)
mse_poly = mean_squared_error(y_test, pred_poly)
rmse_poly = root_mean_squared_error(y_test, pred_poly)

print("Test R²  :", r2_poly)
print("Test MSE:", mse_poly)
print("Test RMSE:", rmse_poly)

In [ ]:
# 6. Demonstration of Underfitting vs Overfitting vs Optimal Fit

# Case 1: Underfitting (Degree 1 / Linear)
p1 = Pipeline([
    ('scale', StandardScaler()),
    ('lr', LinearRegression())
]).fit(X_train, y_train)

# Case 2: Overfitting (Degree 2 Unregularized)
p2_unreg = Pipeline([
    ('scale1', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scale2', StandardScaler()),
    ('lr', LinearRegression())
]).fit(X_train, y_train)

# Case 3: Optimal Fit (Degree 2 + Ridge Regularization alpha=10)
best_ridge = grid.best_estimator_

# Put results in a DataFrame to easily compare performance
fit_comparison = pd.DataFrame({
    'Case': [
        'Degree 1 (Linear)',
        'Degree 2 (Unregularized)',
        'Degree 2 (Ridge Regularized alpha=10)'
    ],
    'Train R2': [
        r2_score(y_train, p1.predict(X_train)),
        r2_score(y_train, p2_unreg.predict(X_train)),
        r2_score(y_train, best_ridge.predict(X_train))
    ],
    'Test R2': [
        r2_score(y_test, p1.predict(X_test)),
        r2_score(y_test, p2_unreg.predict(X_test)),
        r2_score(y_test, pred_poly)
    ],
    'Status': [
        'Underfitting (High Bias)',
        'Overfitting (High Variance)',
        'Optimal Fit (Controlled Variance)'
    ]
})

fit_comparison

## 3. Ridge Regression (L2 Regularization)

In [ ]:
# Pipeline: scaling happens inside, so no manual scaler needed
pipe_ridge = Pipeline([
    ("scale", StandardScaler()),
    ("ridge", Ridge())
])

# Alpha values
alphas_ridge = [0.001, 0.01, 0.1, 1, 10, 100]
param_grid_ridge = {"ridge__alpha": alphas_ridge}

# Cross-validation to pick the best alpha
grid_ridge = GridSearchCV(pipe_ridge, param_grid_ridge, cv=5, scoring="r2", n_jobs=-1)
grid_ridge.fit(X_train, y_train)

# CV score for each alpha
for a, score in zip(alphas_ridge, grid_ridge.cv_results_["mean_test_score"]):
    print(f"alpha={a:<8} CV R² = {score:.4f}")
print("Best alpha:", grid_ridge.best_params_["ridge__alpha"])

alpha=0.001    CV R² = 0.7340
alpha=0.01     CV R² = 0.7340
alpha=0.1      CV R² = 0.7340
alpha=1        CV R² = 0.7338
alpha=10       CV R² = 0.7318
alpha=100      CV R² = 0.7199
Best alpha: 0.1


In [ ]:
# Final evaluation on test data
y_pred_ridge = grid_ridge.predict(X_test)
r2_ridge = r2_score(y_test, y_pred_ridge)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
rmse_ridge = root_mean_squared_error(y_test, y_pred_ridge)

print("R2 Score:", r2_ridge)
print("Test MSE:", mse_ridge)
print("RMSE:", rmse_ridge)

## 4. Lasso Regression (L1 Regularization)

In [ ]:
# Pipeline: scaling happens inside, so no manual scaler needed
pipe_lasso = Pipeline([
    ("scale", StandardScaler()),
    ("lasso", Lasso(max_iter=100000, tol=1e-3))
])

# Alpha values (smaller range suits Lasso)
alphas_lasso = [0.001, 0.01, 0.1, 1]
param_grid_lasso = {"lasso__alpha": alphas_lasso}

# Cross-validation to pick the best alpha
grid_lasso = GridSearchCV(pipe_lasso, param_grid_lasso, cv=5, scoring="r2", n_jobs=-1)
grid_lasso.fit(X_train, y_train)

# CV score for each alpha
for a, score in zip(alphas_lasso, grid_lasso.cv_results_["mean_test_score"]):
    print(f"alpha={a:<8} CV R² = {score:.4f}")
print("Best alpha:", grid_lasso.best_params_["lasso__alpha"])

alpha=0.001    CV R² = 0.7339
alpha=0.01     CV R² = 0.7318
alpha=0.1      CV R² = 0.7092
alpha=1        CV R² = 0.5798
Best alpha: 0.001


In [ ]:
# Final evaluation on test data
y_pred_lasso = grid_lasso.predict(X_test)
r2_lasso = r2_score(y_test, y_pred_lasso)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
rmse_lasso = root_mean_squared_error(y_test, y_pred_lasso)

print("R2 Score:", r2_lasso)
print("Test MSE:", mse_lasso)
print("RMSE:", rmse_lasso)

## 5. Final Comparison: MSE, RMSE, and R² Across All Models

In [ ]:
# Model comparison using variables stored from previous evaluations
models = [
    "Linear Regression (SGD)",
    "Polynomial Regression (Degree 2 + Ridge)",
    "Ridge Regression (alpha=0.1)",
    "Lasso Regression (alpha=0.001)"
]
mse_scores = [mse_linear, mse_poly, mse_ridge, mse_lasso]
rmse_scores = [rmse_linear, rmse_poly, rmse_ridge, rmse_lasso]
r2_scores = [r2_linear, r2_poly, r2_ridge, r2_lasso]

# Summary DataFrame
df_comparison = pd.DataFrame({
    "Model": models,
    "Test MSE": mse_scores,
    "Test RMSE": rmse_scores,
    "Test R2 Score": r2_scores
})
df_comparison = df_comparison.sort_values("Test R2 Score", ascending=False).reset_index(drop=True)
display(df_comparison)

# Visual comparison of models (R2 Score and RMSE)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot R2 Scores (higher is better)
ax1.barh(df_comparison["Model"], df_comparison["Test R2 Score"], color='steelblue')
ax1.set_xlim(0, 1.0)
ax1.set_xlabel("R² Score (Higher is better)")
ax1.set_title("Model Comparison: R² Score")
for i, v in enumerate(df_comparison["Test R2 Score"]):
    ax1.text(v + 0.01, i, f"{v:.4f}", va='center')

# Plot RMSE (lower is better)
ax2.barh(df_comparison["Model"], df_comparison["Test RMSE"], color='coral')
ax2.set_xlabel("RMSE in Kelvin (Lower is better)")
ax2.set_title("Model Comparison: RMSE")
for i, v in enumerate(df_comparison["Test RMSE"]):
    ax2.text(v + 0.3, i, f"{v:.2f}", va='center')

plt.tight_layout()
plt.show()

### Observations:
- **Polynomial Regression (Degree 2 + Ridge, alpha=10)** is the best model: Test R² = 0.8535, RMSE = 12.99 K. It captures non-linear atomic feature interactions missed by linear models.
- **Ridge and Lasso** perform similarly in linear feature space (R² ≈ 0.737). Lasso additionally zeros out irrelevant features, performing implicit feature selection.
- **Linear Regression (Gradient Descent)** converges in 62 SGD iterations with R² = 0.7332.
- **Overfitting** is clearly demonstrated by Degree 2 Unregularized: Train R² = 0.907 vs Test R² = 0.582 (gap > 0.32).
- **Underfitting** is demonstrated by Degree 1 Linear: both Train and Test R² ≈ 0.737, model is too simple.
- **Cross-validation** correctly selected the optimal α for each regularized model, preventing both extremes.